# MemoryBench: LoRA fine-tuning of Qwen2.5-1.5B for fact extraction

Runs on the **free Colab T4 GPU** (Runtime > Change runtime type > T4 GPU), no paid service.

1. Generates the training set (400 synthetic conversations, seed 11, 5 phrasing templates held out so the eval measures generalisation).
2. LoRA fine-tunes `Qwen2.5-1.5B-Instruct` with Unsloth to output the exact JSON the service expects (prompt v2).
3. Exports a **GGUF (q4_k_m)** you download and run locally with Ollama, then evaluate with `python -m evals.run_eval`.

The golden eval set is never used for training.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%pip install -q unsloth
!git clone -q https://github.com/ryannaidji/memorybench.git
%cd memorybench
%pip install -q -r requirements.txt
!python -m evals.gen_train --customers 400 --out finetune/train.jsonl

In [ ]:
from unsloth import FastLanguageModel
import torch, json, time
MAX_LEN = 2048
model, tokenizer = FastLanguageModel.from_pretrained('unsloth/Qwen2.5-1.5B-Instruct', max_seq_length=MAX_LEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=42)
model.print_trainable_parameters()

In [ ]:
from datasets import load_dataset
ds = load_dataset('json', data_files='finetune/train.jsonl', split='train')
def fmt(ex):
    return {'text': tokenizer.apply_chat_template(ex['messages'], tokenize=False)}
ds = ds.map(fmt)
split = ds.train_test_split(test_size=0.1, seed=42)
print(split)
print(split['train'][0]['text'][-400:])

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=split['train'], eval_dataset=split['test'],
    args=SFTConfig(dataset_text_field='text', max_seq_length=MAX_LEN, per_device_train_batch_size=4,
        gradient_accumulation_steps=2, num_train_epochs=2, learning_rate=2e-4, warmup_steps=10,
        logging_steps=10, eval_strategy='epoch', optim='adamw_8bit', seed=42, output_dir='outputs', report_to='none'))
# loss only on the assistant JSON, not on the prompt
trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user
', response_part='<|im_start|>assistant
')
t0 = time.time(); stats = trainer.train(); train_min = (time.time()-t0)/60
print(stats); print(f'train time: {train_min:.1f} min')
print(trainer.evaluate())

In [ ]:
# quick sanity check on a held-out training-split example
FastLanguageModel.for_inference(model)
msgs = json.loads(open('finetune/train.jsonl').readline())['messages'][:2]
inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to('cuda')
out = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.0, do_sample=False)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

In [ ]:
# export GGUF for Ollama (takes a few minutes)
model.save_pretrained_gguf('memorybench-qwen1.5b-ft', tokenizer, quantization_method='q4_k_m')
import glob, os
gguf = [f for f in glob.glob('memorybench-qwen1.5b-ft*/*.gguf') + glob.glob('*.gguf') if 'q4_k_m' in f.lower()]
print(gguf, [round(os.path.getsize(f)/1e9, 2) for f in gguf])
json.dump({'train_minutes': round(train_min, 1), 'train_loss': stats.training_loss, 'eval': trainer.evaluate(),
           'gpu': torch.cuda.get_device_name(0), 'examples': len(split['train'])}, open('finetune_run.json', 'w'), indent=2)
print(open('finetune_run.json').read())

In [ ]:
from google.colab import files
files.download(gguf[0])
files.download('finetune_run.json')

## Then, on your machine

Put the downloaded `.gguf` in `memorybench/finetune/` **renamed to `model.gguf`**, plus `finetune_run.json`, then:

```bash
cd finetune
ollama create memorybench-qwen1.5b-ft -f Modelfile
cd ..
python -m evals.run_eval --model memorybench-qwen1.5b-ft --out evals/results/qwen2.5-1.5b-lora_v2.json
```